In [1]:
from bs4 import BeautifulSoup
import requests
import cloudscraper
import time
import random

In [2]:
scraper = cloudscraper.create_scraper()

In [3]:
def get_number_of_weeks(season: int) -> int | None:
    """Return the number of weeks in a season."""
    week = 1
    while True:
        url = f"https://www.pro-football-reference.com/years/{season}/week_{week}.htm"
        response = scraper.get(url)
        # Stop if the page does not exist (404 or other errors)
        if response.status_code != 200:
            break
        # Stop if the page is a preview for a future week
        title = response.text.split("<title>")[1].split("</title>")[0]
        if "Preview" in title:
            break
        week += 1
        time.sleep(random.uniform(1, 3))
    completed_weeks = week - 1
    return completed_weeks if completed_weeks > 0 else None

In [4]:
def get_week_links(season: int, week: int) -> list[str]:
    """Get the links to all games in a week."""
    week_links = []
    url = f"https://www.pro-football-reference.com/years/{season}/week_{week}.htm"
    response = scraper.get(url)
    soup = BeautifulSoup(response.text, "lxml")
    soup_links = soup.find_all("td", class_="gamelink")
    for l in soup_links:
        link = l.a["href"]
        full_link = f"https://www.pro-football-reference.com{link}"
        week_links.append(full_link)
    return week_links

In [5]:
def get_season_links(season: int):
    """Get the links to all games in a season."""
    season_links = []
    number_of_weeks = get_number_of_weeks(season)
    if number_of_weeks is None:
        return season_links
    for week in range(1, number_of_weeks + 1):
        week_links = get_week_links(season, week)
        season_links.extend(week_links)
        time.sleep(random.uniform(1, 3))
    return season_links

In [6]:
test_links = get_season_links(2010)
len(test_links) #Should be 267 games

267

In [ ]:
### TO-DO: Scrape individual game pages, Aggregate and save

In [9]:
url = test_links[0]
response = scraper.get(url)
soup = BeautifulSoup(response.text, "lxml")

In [17]:
url

'https://www.pro-football-reference.com/boxscores/201009090nor.htm'

In [16]:
soup

<!DOCTYPE html>
<html class="no-js" data-root="/home/pfr/build" data-version="klecko-" lang="en">
<head>
<meta charset="utf-8"/>
<meta content="ie=edge" http-equiv="x-ua-compatible"/>
<meta content="width=device-width, initial-scale=1.0, maximum-scale=2.0" name="viewport"/>
<link href="https://cdn.ssref.net/req/202508071" rel="dns-prefetch"/>
<script>
/* https://docs.osano.com/hc/en-us/articles/22469433444372-Google-Consent-Mode-v2  */
  window.dataLayer = window.dataLayer ||[];
      function gtag(){dataLayer.push(arguments);}
      gtag('consent','default',{
        'ad_storage':'denied',
        'analytics_storage':'denied',
        'ad_user_data':'denied',
        'ad_personalization':'denied',
        'personalization_storage':'denied',
        'functionality_storage':'granted',
        'security_storage':'granted',
        'wait_for_update': 500
      });
      gtag("set", "ads_data_redaction", true);
</script>
<script src="https://cmp.osano.com/16CGnCU8UtNhM14sg/12669873-8cf8-41